# Análisis Avanzado del Pipeline AGI

Este notebook profundiza en el análisis del pipeline con:
- Optimización de hiperparámetros
- Análisis de escalabilidad
- Métricas avanzadas de rendimiento
- Visualizaciones interactivas

In [ ]:
# Configuración inicial (similar a 01_baseline.ipynb)
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

sys.path.append(str(Path.cwd().parent))
from src.models.dummy_agi import DummyAGI, Task, TaskType
from src.pipeline.run_pipeline import TrialRunner, ICPCalculator

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Configuración completada")
print(f"Plotly version: {px.__version__}")

In [ ]:
# Función para experimentos paramétricos
def run_parameter_experiment(param_name, param_values, n_trials_per_value=10):
    """Ejecuta experimentos variando un parámetro"""
    results = []
    
    for value in param_values:
        print(f"Probando {param_name}={value}...")
        
        # Configurar AGI con el parámetro
        config = {param_name: value}
        agi = DummyAGI(config)
        icp_calc = ICPCalculator()
        runner = TrialRunner(agi, icp_calc)
        
        # Ejecutar trials
        trials = runner.run_batch(n_trials_per_value, verbose=False)
        
        # Recopilar estadísticas
        for trial in trials:
            results.append({
                param_name: value,
                'task_type': trial.task.task_type.value,
                'complexity': trial.task.complexity,
                'icp': trial.performance_metrics.get('icp', 0),
                'success': trial.success,
                'execution_time': trial.execution_time
            })
    
    return pd.DataFrame(results)

print("✅ Función de experimentos definida")

In [ ]:
# Experimento: Variación de learning_rate
learning_rates = [0.001, 0.005, 0.01, 0.02, 0.05]
df_lr = run_parameter_experiment('learning_rate', learning_rates, n_trials_per_value=15)

# Visualización interactiva
fig = px.box(df_lr, x='learning_rate', y='icp', color='task_type',
             title='Impacto del Learning Rate en ICP',
             labels={'learning_rate': 'Learning Rate', 'icp': 'ICP Score'})
fig.show()

In [ ]:
# Experimento: Variación de accuracy_base
accuracy_bases = [0.7, 0.8, 0.85, 0.9, 0.95]
df_acc = run_parameter_experiment('accuracy_base', accuracy_bases, n_trials_per_value=15)

# Análisis de escalabilidad
fig = make_subplots(rows=1, cols=2, subplot_titles=['ICP vs Accuracy Base', 'Success Rate vs Accuracy Base'])

# Gráfico 1: ICP
for task_type in df_acc['task_type'].unique():
    task_data = df_acc[df_acc['task_type'] == task_type]
    means = task_data.groupby('accuracy_base')['icp'].mean()
    fig.add_trace(go.Scatter(x=means.index, y=means.values, mode='lines+markers',
                             name=task_type), row=1, col=1)

# Gráfico 2: Success Rate
success_rates = df_acc.groupby('accuracy_base')['success'].mean()
fig.add_trace(go.Bar(x=success_rates.index, y=success_rates.values,
                     name='Success Rate', marker_color='green'), row=1, col=2)

fig.update_layout(height=500, showlegend=True, title_text="Efecto de Accuracy Base")
fig.show()

In [ ]:
# Análisis de escalabilidad con tamaño de datos
print("\n📊 Análisis de escalabilidad")
print("=" * 60)

data_sizes = [100, 500, 1000, 5000, 10000]
scalability_results = []

for size in data_sizes:
    print(f"Probando data_size={size}...")
    agi = DummyAGI()
    icp_calc = ICPCalculator()
    runner = TrialRunner(agi, icp_calc)
    
    for i in range(10):
        task = runner.generate_task(i)
        task.data_size = size  # Sobrescribir tamaño
        result = runner.run_trial(i, task)
        
        scalability_results.append({
            'data_size': size,
            'task_type': task.task_type.value,
            'complexity': task.complexity,
            'execution_time': result.execution_time,
            'icp': result.performance_metrics.get('icp', 0)
        })

df_scalability = pd.DataFrame(scalability_results)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tiempo de ejecución
for task_type in df_scalability['task_type'].unique():
    task_data = df_scalability[df_scalability['task_type'] == task_type]
    means = task_data.groupby('data_size')['execution_time'].mean()
    axes[0].plot(means.index, means.values, marker='o', label=task_type, linewidth=2)

axes[0].set_xlabel('Data Size')
axes[0].set_ylabel('Execution Time (s)')
axes[0].set_title('Escalabilidad: Tiempo vs Tamaño de Datos')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ICP
for task_type in df_scalability['task_type'].unique():
    task_data = df_scalability[df_scalability['task_type'] == task_type]
    means = task_data.groupby('data_size')['icp'].mean()
    axes[1].plot(means.index, means.values, marker='s', label=task_type, linewidth=2)

axes[1].set_xlabel('Data Size')
axes[1].set_ylabel('ICP Score')
axes[1].set_title('Escalabilidad: ICP vs Tamaño de Datos')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()